In [1]:
import os

from langchain_ollama import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model='llama3.2:1b')

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

In [6]:
# Load the document, split it into chunks, embed each chunk and load it into the vector store.

raw_documents = TextLoader("speech.txt").load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

documents = text_splitter.split_documents(raw_documents)

## Chroma
Chroma is an open-source vector database designed for storing, indexing, and querying high-dimensional vector embeddings. It’s optimized for fast similarity searches, making it a popular choice in AI applications like semantic search, recommendation systems, and chatbots.

In [5]:
!pip install langchain_chroma -q


[notice] A new release of pip available: 22.3 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_chroma import Chroma

db = Chroma.from_documents(documents, embeddings_model)

In [19]:
## We can aslo store like this
vector_store = Chroma(collection_name="chatbot_memory", embedding_function=embeddings_model)

# Add Documents
texts = ["LangChain simplifies LLM app development.", 
         "Chroma is an efficient vector database."]

vector_store.add_texts(texts)

['7341233d-b97e-4332-8f7f-1075d1d0eaeb',
 '21e555a5-a802-44db-9cba-10f7dcbb1b7c']

### Similarity search
All vectorstores expose a similarity_search method. This will take incoming documents, create an embedding of them, and then find all documents with the most similar embedding.

In [10]:
query = "To such a task we can dedicate our lives"

docs = db.similarity_search(query)

In [14]:
# docs
# docs[0]
docs[0].page_content

'To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'

In [17]:
# for i, doc in enumerate(docs):
#     print(f"Result {i+1}: {doc.page_content}")

### Similarity search by vector

In [22]:
embedding_vector = OllamaEmbeddings(model='llama3.2:1b').embed_query(query)
docs = db.similarity_search_by_vector(embedding_vector)
print(docs[0].page_content)

To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.


### Async Operations

In [25]:
docs = await db.asimilarity_search(query)
docs[0].page_content

'To such a task we can dedicate our lives and our fortunes, everything that we are and everything that we have, with the pride of those who know that the day has come when America is privileged to spend her blood and her might for the principles that gave her birth and happiness and the peace which she has treasured. God helping her, she can do no other.'

## FAISS (Facebook AI Similarity Search)
FAISS is an open-source library developed by Facebook AI Research for efficient similarity search and clustering of dense vectors. It's designed to handle large-scale vector data with high speed and low memory usage, making it perfect for applications like semantic search, recommendation systems, and LLM-based chatbots.

In [27]:
!pip install faiss-cpu -q


[notice] A new release of pip available: 22.3 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(documents, embeddings_model)

### Similarity search

In [33]:
query = "To such a task we can dedicate our lives"

docs = db.similarity_search(query)

print(docs[0].page_content)

The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.

Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.

â€¦


### Similarity search by vector

In [34]:
embedding_vector = embeddings_model.embed_query(query)

docs = db.similarity_search_by_vector(embedding_vector)

print(docs[0].page_content)

The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.

Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.

â€¦


In [36]:
docs = await db.asimilarity_search(query)
print(docs[0].page_content)

The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.

Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.

â€¦


## Lance

In [38]:
!pip install lancedb -q

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\saraw\Documents\python-env\venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 437, in _error_catcher
    yield
  File "C:\Users\saraw\Documents\python-env\venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 560, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\saraw\Documents\python-env\venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 526, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\saraw\Documents\python-env\venv\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 90, in read
    data = self.__fp.read(amt)
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\saraw\AppData\Local\Programs\Python\Python311\Lib\http\client.py", line 465, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "C:\Users\saraw\AppData\Local\P

In [39]:
from langchain_community.vectorstores import LanceDB
import lancedb

ModuleNotFoundError: No module named 'lancedb'

In [40]:
db = db.connect("/tmp/lancedb")

table = db.create_table(
    "mt_table",
    data=[
        {
            "vector": embeddings.embed_query("Hello World"),
            "text": "Hello World",
            "id": "1",
        }
    ],
    mode = "overwrite",
)
db = LanceDB.from_documents(documents, embeddings_model)

AttributeError: 'FAISS' object has no attribute 'connect'

### Similarity search

In [ ]:
query = "To such a task we can dedicate our lives"

docs = db.similarity_search(query)

print(docs[0].page_content)